# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   null|     BE|   null|    null|      1|  null|   269|  6|    69| null|       1|    null|    0.0|    null|    null|    null|    null|    null|    null|    null|
|3070802| 1963| 1096|   null|     US|     TX|    null|      1|  null|     2|  6|    63| null|       0|    null|   null|    null|    null|    null|    null|    null|    null|    null|
|3070803| 1963| 1096|   null|     US|     IL|    null|      1|  null|     2|  6|    6

Adding some alias names for the patent table for the joining later

In [ ]:
p_cited = patents.alias("P_CITED")
p_citing = patents.alias("P_CITING")

Doing the inner joins for both the citing and cited patent codes using the aliases set above.

In [ ]:
matched = citations.join(
    p_cited,
    citations["CITED"] == p_cited["PATENT"],
).join(
    p_citing,
    citations["CITING"] == p_citing["PATENT"],
)

Import the rest of items that would be useful to get the sql style query ready. The filter var is just making sure that we only get the elements that have something in the state column for later ues. I made sure that the element is not null nor is the empty string literal.

In [ ]:
from pyspark.sql.functions import count, coalesce, lit, col
filtered = matched.filter(
    (col("P_CITED.POSTATE").isNotNull()) & (col("P_CITING.POSTATE").isNotNull()) &\
        (col("P_CITING.POSTATE") != lit("")) & (col("P_CITED.POSTATE") != lit("")) & \
        (col("P_CITED.POSTATE") == col("P_CITING.POSTATE"))
)

Then I am making sure that I am grouping based on the citing and aggregate the count on the cited patent. The count is set the alias as same_State_count to have it be able to access.

In [ ]:
same_State_counts = filtered\
    .groupBy("CITING")\
    .agg(count(p_cited["PATENT"]).alias("same_State_count"))

To get the result, I do the left join on the patent table with relation to the counts. The join is based on whether the patent is the same as the same_State_count citing so that the result table has the correct count. After the left join, I select everything in the patent table and made sure that makes a list of arguments from left to right and returns the very first non-NULL value it encounters. I rename the same_State_count to CO_STATE and ordered the whole table by that count in descending order.

In [ ]:
result = patents.join(
    same_State_counts,
    patents["PATENT"] == same_State_counts["CITING"],
    how="left"
).select(
    patents["*"],
    coalesce(
        same_State_counts["same_State_count"],
        lit(0)
    ).alias("CO_STATE"),
).orderBy(col("CO_STATE").desc())
return result

To see the results, I have a main python file that imports the function that has all of the code from the dataframe api and the query and shows the dataframe. The rdd import is for the other element part of this assignment.

In [ ]:
from dataframe import patent_DataFrame
from rdd import patent_RDD
def main():
    df = patent_DataFrame()
    df.show(5)

if __name__ == "__main__":
    main()